# Imports

In [57]:
import json
from tqdm import tqdm
from py3langid.langid import LanguageIdentifier, MODEL_FILE
import csv

# Data

In [8]:
with open('../Data/test_KTH_only.json', 'r') as file:
    data = json.load(file)

print(type(data), len(data))

<class 'list'> 2435


# Clean keywords

In [32]:
identifier = LanguageIdentifier.from_pickled_model(MODEL_FILE, norm_probs=True) 
for row in tqdm(data):
    keywords = row['Keywords']
    new_keywords = []
    for kw in keywords:
        if kw == '':
            continue
        # Get language probabilities
        lang_probs = identifier.rank(kw)
        en_prob = next((prob for lang, prob in lang_probs if lang == 'en'), 0)
        sv_prob = next((prob for lang, prob in lang_probs if lang == 'sv'), 0)
        if en_prob >= sv_prob:
            new_keywords.append(kw.lower())
    row['Keywords'] = new_keywords

100%|██████████| 2435/2435 [01:05<00:00, 37.41it/s]


In [41]:
# Count non-empty rows
counter = 0
unique_keywords = set()
for row in data:
    if row['Keywords']:
        counter += 1
        for kw in row['Keywords']:
            unique_keywords.add(kw)
        print(row['Keywords'])
print(counter, 'rows with keywords out of', len(data))

['computer vision', '3d reconstruction', 'text-to-3d', 'gaussian splatting', 'diffusion models', 'deep learning', 'datorseende', 'gaussian splatting', 'diffusion modeller']
['3d machine learning', 'computer vision', 'shape', 'point cloud']
['3d shape retrieval', 'pattern recognition', 'machine learning', 'autoencoder', 'k-nn', '3d shape retrieval', 'pattern recognition', 'machine learning', 'autoenconder', 'k-nn']
['computer vision', 'machine learning', 'autonomous vehicles', 'autonomous cars', 'lider', 'point cloud', 'deep learning', 'yolo']
['3d/2d', 'image registration', 'radiosurgery']
['cnn', 'svm', 'dbscan', 'mimo', 'bandwidth', 'frequencies', 'human presence detection', 'cnn', 'svm', 'dbscan', 'mimo', 'bandbredd']
['information visualization', ' augmented project based learning', ' large public demos']
['security', 'privacy', 'vehicular pki', 'vpki', 'identity and credential management', 'vehicular communications', 'vanets', 'availability', 'scalability', 'resilient', 'efficienc

# New dataset for zero-shot classification

In [60]:
fields = ['PID', 'Title', 'Keywords', 'Content']
with open('zeroshot.csv', 'w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=fields, quoting=csv.QUOTE_ALL)
    writer.writeheader()
    for row in data:
        row = {key: row[key] for key in fields}
        writer.writerow(row)